# Optimal Steady-State Reactor Design

An optimal process design problem originally presented by [[1](#references)] that involves a continuous stirred-tank reactor (CSTR) and separator train with recycle for the chlorination of benzene with the following reactions taking place:
$$
    \begin{array}{rcl}
    \text{C}_6\text{H}_6 + \text{Cl}_2 &\rightarrow& \text{C}_6\text{H}_5\text{Cl} + \text{HCl}\\
    \text{C}_6\text{H}_5\text{Cl} + \text{Cl}_2 &\rightarrow& \text{C}_6\text{H}_4\text{Cl}_2 + \text{HCl}\\
    \end{array}
$$
where the rate constants $k_1$ and $k_2$ [h $^{-1}$] are known and the reactor volume $V_{\text{cstr}}$ [m $^3$] and feed flowrate $F_{\text{feed}}$ [kmol/h] are free design variables. The CSTR is followed by a separation train for product purification and reactant recycle as illustrated in the figure below.

<div style="text-align: center;">
  <img src="CSTR.png" width="75%" height="75%">
</div>

For simplicity, the reactions are considered first-order (no dependence on $\text{Cl}_2$) with respect to benzene ($A$) and chlorobenzene ($B$), similar to the treatment by [[1](#references)]. The molar volumes of each species are: $V_A = 8.937 \times 10^{-2}$ m $^3$/kmol, $V_B = 1.018 \times 10^{-1}$ m $^3$/kmol, and $V_C=1.13\times10^{-1}$ m $^3$/kmol, with dichlorobenzene represented as species $C$. The feed is considered to be pure $A$. The rate constants are $k_1=0.40$ h $^{-1}$ and $k_2=0.055$ h $^{-1}$. The mole fraction of $i ∈ \{A, B, C \}$ in stream $j \in S$ is defined as $y_{i,j}$ and $r_1$ and $r_2$ are the reaction rates [kmol m $^{-3}$ h $^{-1}$] defined as:
$$
    \begin{aligned}
        r_1 = k_1\dfrac{y_{A,\text{cstr.out}}}{y_{A,\text{cstr.out}}V_A + y_{B,\text{cstr.out}}V_B + y_{C,\text{cstr.out}}V_C},\\
        r_2 = k_2\dfrac{y_{B,\text{cstr.out}}}{y_{A,\text{cstr.out}}V_A + y_{B,\text{cstr.out}}V_B + y_{C,\text{cstr.out}}V_C}.
    \end{aligned}
$$

In [ ]:
using ModelingToolkit, JuMP, EAGO, EOptInterface
using ModelingToolkit: t_nounits as t, D_nounits as D

The system is modeling in `ModelingToolkit` by defining a `ModelingToolkit.@mtkmodel` for each component (or unit operation) illustrated in the figure and a `ModelingToolkit.@connector` is defined for the variables in the stream.

In [2]:
@connector Stream begin
    @variables begin
        F(t),   [input=true]
        y_A(t), [input=true]
        y_B(t), [input=true]
        y_C(t), [input=true]
    end
    @parameters begin
        V_A = 8.937e-2
        V_B = 1.018e-1
        V_C = 1.13e-1
    end
end
@mtkmodel Influent begin
    @components begin
        out = Stream()
    end
    @parameters begin
        # Unknown parameters (free design variables)
        F

        # Known parameters
        y_A = 1.0
        y_B = 0.0
        y_C = 0.0
    end
    @equations begin
        out.F ~ F
        out.y_A ~ y_A
        out.y_B ~ y_B
        out.y_C ~ y_C
    end
end
@mtkmodel Mixer begin
    @components begin
        in1 = Stream()
        in2 = Stream()
        out = Stream()
    end
    @equations begin
        out.F ~ in1.F + in2.F
        out.y_A ~ (in1.y_A*in1.F + in2.y_A*in2.F)/(in1.F + in2.F)
        out.y_B ~ (in1.y_B*in1.F + in2.y_B*in2.F)/(in1.F + in2.F)
        out.y_C ~ (in1.y_C*in1.F + in2.y_C*in2.F)/(in1.F + in2.F)
    end
end
@mtkmodel CSTR begin
    @components begin
        in = Stream()
        out = Stream()
    end
    @parameters begin
        # Unknown parameters (free design variables)
        V

        # Known parameters
        k_1 = 0.4
        k_2 = 0.055
    end
    begin
        r_1 = k_1*out.y_A/(out.y_A*in.V_A + out.y_B*in.V_B + out.y_C*in.V_C)
        r_2 = k_2*out.y_B/(out.y_A*in.V_A + out.y_B*in.V_B + out.y_C*in.V_C)
    end
    @equations begin
        out.F ~ in.F
        out.y_A + out.y_B + out.y_C ~ 1.0
        out.y_B*out.F ~ in.y_B*in.F + (r_1 - r_2)*V
        out.y_C*out.F ~ in.y_C*in.F + r_2*V
    end
end
@mtkmodel Separator1 begin
    @components begin
        in = Stream()
        outV = Stream()
        outL = Stream()
    end
    @equations begin
        in.F ~ outV.F + outL.F
        in.y_B*in.F ~ outL.y_B*outL.F
        in.y_C*in.F ~ outL.y_C*outL.F
        
        outV.y_A + outV.y_B + outV.y_C ~ 1.0
        outV.y_C ~ 0.0
        outV.y_B ~ 0.0

        outL.y_A + outL.y_B + outL.y_C ~ 1.0
        outL.y_A ~ 0.0
    end
end
@mtkmodel Separator2 begin
    @components begin
        in = Stream()
        outV = Stream()
        outL = Stream()
    end
    @equations begin
        in.F ~ outV.F + outL.F
        in.y_B*in.F ~ outV.F

        outV.y_A + outV.y_B + outV.y_C ~ 1.0
        outV.y_A ~ 0.0
        outV.y_C ~ 0.0

        outL.y_A + outL.y_B + outL.y_C ~ 1.0
        outL.y_A ~ 0.0
        outL.y_B ~ 0.0
    end
end;

The components are then connected together in a final `ModelingToolkit.System` via `ModelingToolkit.@equations` to define the full system. Note that we use `ModelingToolkit.@mtkcompile` to build the reduced-space model and `ModelingToolkit.@named` to build the full-space model.

In [3]:
@mtkmodel ReactorSeparatorRecycle begin
    @components begin
        influent = Influent()
        mixer = Mixer()
        cstr = CSTR()
        sep1 = Separator1()
        sep2 = Separator2()
    end
    @equations begin
        connect(influent.out, mixer.in1)
        connect(mixer.out, cstr.in)
        connect(cstr.out, sep1.in)
        connect(sep1.outV, mixer.in2)
        connect(sep1.outL, sep2.in)
    end
end

@mtkcompile system = ReactorSeparatorRecycle()
@named full_system = ReactorSeparatorRecycle();

Next, we define the performance specifications and objective in our optimal design problem. For this example, we must produce at least 25 kmol/h of $B$ and we cannot have a reactor larger than 10 m $^3$ due to space limitations. We also force a lower-bound on the residence time $\tau$ for the reactor of 475 seconds. These specifications provide the inequality constraints
$$
    \mathbf{g} (\mathbf{x}) =
    \begin{bmatrix}
        25 - F_{\text{sep2}.\text{outV}} \\
        \dfrac{475}{3600} - \dfrac{V_{\text{cstr}}}{F_{\text{cstr.out}}(y_{A, \text{cstr.out}} V_A + y_{B, \text{cstr.out}} V_B + y_{C, \text{cstr.out}} V_C)}
    \end{bmatrix} \le \mathbf 0
$$
and an upper bound on $V_{\text{cstr}}$.

We consider an economic objective for our optimization problem as the total annualized costs of the unit operations. The total annualized cost of the CSTR is given by:
$$
    f_{cstr} = (25764 + 8178 V_\text{cstr})/2.5,
$$
and the total annualized cost of the separator train is given by:
$$
    \begin{array}{rcl}
    c_{sep1}^{cap} &=& 132718 + F_\text{sep1.in} (369 y_{A, \text{sep1.in}} - 1113.9 y_{B, \text{sep1.in}} ) \\
    c_{sep2}^{cap} &=& 25000 + F_\text{sep2.in} (6984.5 y_{B, \text{sep2.in}} - 3869.53 y_{C, \text{sep2.in}}^2) \\
    c_{sep1}^{op} &=& F_\text{sep1.in} (3 + 36.11 y_{A, \text{sep1.in}} + 7.71 y_{B,\text{sep1.in}})(26.32 \times 10^{-3}) \\
    c_{sep2}^{op} &=& F_\text{sep2.in} (26.21 + 29.45 y_{B, \text{sep2.in}}) (26.32 \times 10^{-3}) \\
    f_{sep} &=& (c_{sep1}^{cap} + c_{sep2}^{cap})/2.5 + 0.52 (c_{sep1}^{op} + c_{sep2}^{op}).
    \end{array}
$$
The total annualized cost of the complete system is then given by
$$
    f_{total} = f_{cstr} + f_{sep}.
$$
We can define these expressions and objective function symbolically.

In [4]:
exprF5 = system.sep2.outV.F
exprTau = system.cstr.V/(system.cstr.out.F*(system.cstr.out.y_A*system.cstr.in.V_A + system.cstr.out.y_B*system.cstr.in.V_B + system.cstr.out.y_C*system.cstr.in.V_C))
f_CSTR = (25764.0 + 8178.0*system.cstr.V)/2.5
s1cap = 132718.0 + system.cstr.out.F*(369.0*system.cstr.out.y_A - 1113.9*system.cstr.out.y_B)
s2cap = 25000.0 + system.sep1.outL.F*(6984.5*system.sep1.outL.y_B - 3869.53*system.sep1.outL.y_C^2)
s1op = system.cstr.out.F*(3.0 + 36.11*system.cstr.out.y_A + 7.71*system.cstr.out.y_B)*26.32e-3
s2op = system.sep1.outL.F*(26.21 + 29.45*system.sep1.outL.y_B)*26.32e-3
f_Sep = (s1cap + s2cap)/2.5 + 0.52*(s1op + s2op)
g1 = 25.0 - exprF5
g2 = 475/3600 - exprTau
obj = f_CSTR + f_Sep;

We can then define the `JuMP.Model` using an appropriate solver of choice. To solve this problem to global optimality, we have chosen to use `EAGO` [[2](#references)].

In [5]:
model = Model(EAGO.Optimizer)

A JuMP Model
├ solver: EAGO - Easy Advanced Global Optimization
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

We can use `EOptInterface.decision_vars` on the `ModelingToolkit.System` to retrieve the decision variables for the optimization problem.

In [6]:
decision_vars(system)

6-element Vector{Any}:
 sep1₊in₊F(t)
 sep1₊in₊y_B(t)
 sep1₊in₊y_C(t)
 sep1₊outL₊y_C(t)
 influent₊F
 cstr₊V

Now that we know which variables appear in the system, we can add the decision variables to the `JuMP.Model` with appropriate bounds.

In [7]:
xL = zeros(6)
xU = [100.0, 1.0, 1.0, 1.0, 100.0, 10.0]
@variable(model, xL[i] <= x[i=1:6] <= xU[i]);

We then use `EOptInterface.register_nlsystem` to register our `ModelingToolkit.System` constraints and objective function.

In [8]:
register_nlsystem(model, system, obj, [g1, g2])

Next, we optimize the `JuMP.Model` and retrieve the results.

In [9]:
JuMP.optimize!(model)
println("Termination Status: $(JuMP.termination_status(model))")
println("Primal Status: $(JuMP.primal_status(model))")
println("Solve Time: $(round.(JuMP.solve_time(model), digits=5))")
println("f^* = $(round(JuMP.objective_value(model), digits=5))")
println("x* = $(round.(JuMP.value.(x), digits=3))")

┌ Warning: At least one branching variable is unbounded. This will interfere with EAGO's global
│ optimization routine and may cause unexpected results. Bounds have been automatically
│ generated at +/- 1E6 for all unbounded variables, but tighter user-defined bounds are
│ highly recommended. To disable this warning and the automatic generation of bounds, use
│ the option `unbounded_check = false`.
└ @ EAGO C:\Users\ybr24001\.julia\packages\EAGO\WUXx0\src\eago_optimizer\optimize\nonconvex\stack_management.jl:255



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

-----------------------------------------------------------------------------------------------------------------
| Iteration # |    Nodes    | Lower Bound | Upper Bound |     Gap     |    Ratio    |    Timer    |  Time Left  |
-----------------------------------------------------------------------------------------------------------------
|         211 |          26 |   1.697E+05 |   1.699E+05 |   1.639E+02 |   9.650E-04 |       40.27 |     3559.73 |
-----------------------------------------------------------------------------------------------------------------
 
Relative Tolerance Achieved
Optimal Solut

We then use `EOptInterface.full_solution` to calculate the full-space solution of the original `ModelingToolkit.System`.

In [10]:
full_solution(model, system)

Dict{Any, Any} with 44 entries:
  cstr₊in₊F(t)        => 95.0215
  sep2₊outV₊F(t)      => 25.0
  sep2₊in₊y_C(t)      => 0.0500329
  sep1₊outV₊y_C(t)    => 0.0
  influent₊out₊y_B(t) => 0.0
  mixer₊in2₊y_A(t)    => 1.0
  cstr₊out₊y_A(t)     => 0.723045
  mixer₊in2₊F(t)      => 68.7048
  cstr₊in₊y_B(t)      => 0.0
  sep2₊outV₊y_A(t)    => 0.0
  mixer₊out₊y_C(t)    => 0.0
  sep1₊outV₊y_A(t)    => 1.0
  mixer₊in2₊y_B(t)    => -0.0
  mixer₊out₊y_B(t)    => 0.0
  mixer₊in1₊y_A(t)    => 1.0
  cstr₊in₊y_C(t)      => 0.0
  sep2₊in₊y_B(t)      => 0.949967
  sep2₊in₊y_A(t)      => 0.0
  mixer₊out₊F(t)      => 95.0215
  ⋮                   => ⋮

We can then define another `JuMP.Model` to solve the full-space model.

In [11]:
full_model = Model(EAGO.Optimizer)

xL = zeros(50)
xU = vcat(repeat([100.0, 1.0, 1.0, 1.0], 12), 100.0, 10.0)
@variable(full_model, xL[i] <= x[i=1:50] <= xU[i])

register_nlsystem(full_model, full_system, obj, [g1, g2])

JuMP.optimize!(full_model)
println("Termination Status: $(JuMP.termination_status(full_model))")
println("Primal Status: $(JuMP.primal_status(full_model))")
println("Solve Time: $(round.(JuMP.solve_time(full_model), digits=5))")
println("f^* = $(round(JuMP.objective_value(full_model), digits=5))")
println("x* = $(round.(JuMP.value.(x), digits=3))")

-----------------------------------------------------------------------------------------------------------------
| Iteration # |    Nodes    | Lower Bound | Upper Bound |     Gap     |    Ratio    |    Timer    |  Time Left  |
-----------------------------------------------------------------------------------------------------------------
|        1000 |         131 |   1.262E+05 |   1.699E+05 |   4.370E+04 |   2.572E-01 |       28.14 |     3571.86 |
|        2000 |         323 |   1.438E+05 |   1.699E+05 |   2.612E+04 |   1.537E-01 |       37.29 |     3562.71 |
|        3000 |         445 |   1.497E+05 |   1.699E+05 |   2.017E+04 |   1.187E-01 |       45.34 |     3554.66 |
|        4000 |         505 |   1.524E+05 |   1.699E+05 |   1.746E+04 |   1.028E-01 |       53.10 |     3546.90 |
|        5000 |         555 |   1.542E+05 |   1.699E+05 |   1.565E+04 |   9.211E-02 |       60.99 |     3539.01 |
|        6000 |         639 |   1.555E+05 |   1.699E+05 |   1.437E+04 |   8.458E-02 |   

We can see the results are the same, but the reduced-space model takes significantly less time to solve to global optimality.

### References

1. Kokossis, A.C. and Floudas, C.A. Synthesis of isothermal reactor-separator-recycle systems. *Chemical Engineering Science.* 46, 1361-1383 (1991). DOI: [10.1016/0009-2509(91)85063-4](https://doi.org/10.1016/0009-2509(91)85063-4)
2. Wilhelm, M. E. and Stuber, M.D. EAGO.jl: easy advanced global optimization in Julia. *Optimization Methods and Software.* 37(2), 425-450 (2022). DOI: [10.1080/10556788.2020.1786566](https://doi.org/10.1080/10556788.2020.1786566)